In [1]:
# import
import sys
import os
import time
import urllib.request
import numpy as np

# from
from pathlib import Path
from bs4 import BeautifulSoup
from PIL import Image
from io import BytesIO

# selenium
from selenium import webdriver
from selenium.webdriver.common.keys import Keys
from selenium.webdriver import ActionChains

# tesseract
import cv2
import pytesseract

In [2]:
url = "https://cvmweb.cvm.gov.br/SWB//Sistemas/SCW/CPublica/CConsolFdo/FormBuscaParticFdo.aspx"

In [3]:
# Opções para navegação (ver Chrome Options na documentação do Selenium)
Options = webdriver.ChromeOptions()
# Options.add_argument('--headless')              # evita que abra uma janela  (necessário para funcionar no Colab)
Options.add_argument("--start-maximized")       # inicia a janela maximizada
Options.add_argument('--no-sandbox')
Options.add_argument('--disable-dev-shm-usage')
prefs = {"download.default_directory" : '/fotos'}
Options.add_experimental_option("prefs", prefs) # define opções de preferência (ex: pasta para download)

carteira_cnpj = []

for i in range(0,2):
    ativos_i = []

    driver = webdriver.Chrome(options= Options)
    driver.get(url)

    while len(driver.current_url)<140:
        # acessa o site
        print('(0) Entrando no site...')
        driver.get(url)
        driver.implicitly_wait(3)

        # Baixando o arquivo de imagem contendo o número:
        print('(1) Baixando a imagem...')
        img = driver.find_element_by_xpath('//*[@id="trRandom3"]/td[2]/img')
        png = driver.get_screenshot_as_png()
        im = Image.open(BytesIO(png))
        location = img.location
        size = img.size
        left = location['x']
        top = location['y']
        right = location['x'] + size['width']
        bottom = location['y'] + size['height']
        im = im.crop((left, top, right, bottom)) # defines crop points
        im.save('number.png')                   # saves new cropped image
        # src = img.get_attribute('src')
        # urllib.request.urlretrieve(src,'number.jpeg')
        print('(2) Imagem Baixada.')
        driver.implicitly_wait(3)

        # Descobrindo o Número da Imagem:
        image = cv2.imread('number.png', 0)
        thresh = cv2.threshold(image, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)[1]
        data = pytesseract.image_to_string(thresh, lang='eng',config='--psm 10 --oem 3 -c tessedit_char_whitelist=0123456789')
        print('(3) O Número é: ' + str(data))
        driver.implicitly_wait(2)

        # Adicionando o número para os dados de entrada:
        login={'numero':data}
        driver.implicitly_wait(3)

        # Inserindo o Número:
        print('Inserindo o Número...')
        number = driver.find_element_by_xpath('//*[@id="numRandom"]')
        number.send_keys(login['numero'])
        driver.implicitly_wait(5)

    print("Entrou com Sucesso!")

    # Entrando no CNPJ:
    if i%2==0:
        cnpj = '//*[@id="ddlFundos__ctl'+str(i)+'_Linkbutton2"]'
    else:
        cnpj = '//*[@id="ddlFundos__ctl'+str(i)+'_Linkbutton3"]'
    cnpj_number = driver.find_element_by_xpath(cnpj)
    print("Entrando no CNPJ: " + str(cnpj_number.text))
    cnpj_i = str(cnpj_number.text)
    cnpj_number.click()
    driver.implicitly_wait(3)

    # Entrando na Composição da Carteira:
    carteira_url = driver.find_element_by_xpath('//*[@id="Hyperlink1"]/font').click()

    # Pegando a composição da carteira:
    for j in range(5,10):
        try:
            xpath = '//*[@id="dlAplics"]/tbody/tr['+str(j)+']/td[1]/span'
            composicao = driver.find_element_by_xpath(xpath)
            print(composicao.text)
            ativos_i.append(str(composicao.text))
        except:
            pass

    # Fazendo o dicionário:
    cart = {cnpj_i: ativos_i}
    carteira_cnpj.append(cart)

    #Fechando o driver:
    driver.close()

(0) Entrando no site...
(1) Baixando a imagem...
(2) Imagem Baixada.
(3) O Número é: 
Inserindo o Número...
(0) Entrando no site...
(1) Baixando a imagem...
(2) Imagem Baixada.
(3) O Número é: 7

Inserindo o Número...
(0) Entrando no site...
(1) Baixando a imagem...
(2) Imagem Baixada.
(3) O Número é: 2288

Inserindo o Número...
(0) Entrando no site...
(1) Baixando a imagem...
(2) Imagem Baixada.
(3) O Número é: 7

Inserindo o Número...
(0) Entrando no site...
(1) Baixando a imagem...
(2) Imagem Baixada.
(3) O Número é: 
Inserindo o Número...
(0) Entrando no site...
(1) Baixando a imagem...
(2) Imagem Baixada.
(3) O Número é: 99644

Inserindo o Número...
(0) Entrando no site...
(1) Baixando a imagem...
(2) Imagem Baixada.
(3) O Número é: 
Inserindo o Número...
(0) Entrando no site...
(1) Baixando a imagem...
(2) Imagem Baixada.
(3) O Número é: 58213

Inserindo o Número...
(0) Entrando no site...
(1) Baixando a imagem...
(2) Imagem Baixada.
(3) O Número é: 
Inserindo o Número..

In [4]:
print(carteira_cnpj)

[{'34.172.417/0001-53': ['Cotas de Fundos\nDYNAMO COUGAR FUNDO DE INVESTIMENTO EM AÇÕES', 'Cotas de Fundos\nATMR II FUNDO DE INVESTIMENTO EM COTAS DE FUNDOS DE INVESTIMENTO DE AÇÕES', 'Cotas de Fundos\nINDIE FUNDO DE INVESTIMENTO EM COTAS DE FUNDOS DE INVESTIMENTO DE AÇÕES', 'Cotas de Fundos\nHIX CAPITAL FUNDO DE INVESTIMENTO EM COTAS DE FUNDOS DE INVESTIMENTO EM AÇÕES', 'Cotas de Fundos\nVISTA FUNDO DE INVESTIMENTO EM COTAS DE FUNDOS DE INVESTIMENTO DE AÇÕES']}, {'36.016.411/0001-12': ['Cotas de Fundos\n051 CP ALLOCATION FI EM COTAS DE FUNDOS DE INVESTIMENTO MULTIMERCADO CREDITO PRIVADO', 'Cotas de Fundos\n051 HF FUNDO DE INVESTIMENTO EM COTAS DE FUNDOS DE INVESTIMENTO MULTIMERCADO CREDITO PRIVADO', 'Cotas de Fundos\n051 AÇÕES FUNDO DE INVESTIMENTO DE AÇÕES', 'Cotas de Fundos\nBTG PACTUAL TESOURO SELIC FUNDO DE INVESTIMENTO RENDA FIXA REFERENCIADO DI', 'Cotas de Fundos\nBTG PACTUAL OURO FUNDO DE INVESTIMENTO EM COTAS DE FUNDOS DE INVESTIMENTO MULTIMERCADO']}]
